# Benchmark: MLSTRUCT-FP / U-Net (Pizarro) contra el ground truth propio (PDV Nivel 1 y Nivel 2)

Tercer y último modelo del benchmark de extracción geométrica (después de Raster2Seq y MitUNet, ver roadmap). Usa el checkpoint pre-entrenado `no_rot_256_50` del propio equipo de Pizarro -- el runbook (`Fase 2/P1b_Runbook_UNet_MLSTRUCT-FP.md`) recomendaba probarlo antes de gastar en entrenamiento propio, nunca se había corrido.

**Runtime necesario: GPU** (Entorno de ejecución → Cambiar tipo de entorno → GPU).

**Aviso honesto**: a diferencia de MitUNet/Raster2Seq (PyTorch), este repo usa TensorFlow/Keras con un framework de investigación propio (formato de 'sesión' para el checkpoint, no un .h5 suelto) y el propio notebook de los autores nunca muestra inferencia sobre una imagen custom -- se construyó llamando directo a la clase `UNETFloorPhotoModel`/`predict()` leyendo el código fuente real (`_model.py`, `_fp_unet.py`), no por prueba y error como con Raster2Seq. Aun así, es la primera vez que se corre -- probable que necesite ajustes en vivo.

**Qué subir cuando lo pida la Celda 2**: las mismas 2 imágenes de siempre -- `archicheck_geometrico_pdv_05ago_0009_pag2-1.png` (Nivel 1) y `_pag2-2.png` (Nivel 2), en `Fase 2/Desarrollos/Test/pdv/old/`.


## Celda 1 — Clonar los 2 repos e instalar dependencias


In [ ]:
%cd /content
!git clone https://github.com/MLSTRUCT/MLSTRUCT-FP.git
!git clone https://github.com/MLSTRUCT/MLStructFP_benchmarks.git
%cd /content/MLStructFP_benchmarks

# El README pide Python 3.8 + CUDA 10.1, pero eso es para ENTRENAR (build
# environment viejo, 2023). Para correr solo inferencia con un checkpoint ya
# entrenado, un TF moderno debería alcanzar (el modelo es un U-Net estandar,
# sin ops CUDA custom) -- se intenta primero con el TF que ya trae Colab antes
# de forzar una version vieja.
# Fix real 2026-08-06 (segunda vuelta): 'pip install -e .' rechaza instalar
# el paquete porque su propio setup.py declara python_requires='>=3.8,<3.9'
# -- Colab tiene Python 3.12. Es codigo Python puro (no C/CUDA), asi que en
# vez de pelear con pip se agrega la carpeta directo al sys.path en la Celda
# 5 (equivalente a lo que haria una instalacion editable, sin el chequeo de
# version que la bloquea).
!pip install -q gdown


## Celda 2 — Subir las 2 imágenes de prueba


In [ ]:
from google.colab import files
import os

os.makedirs('/content/test_images', exist_ok=True)
print('Selecciona las 2 imagenes (Nivel 1 y Nivel 2) cuando aparezca el boton:')
uploaded = files.upload()
for name, data in uploaded.items():
    with open(f'/content/test_images/{name}', 'wb') as f:
        f.write(data)

NIVEL1_PATH = f'/content/test_images/{[n for n in uploaded if "pag2-1" in n][0]}'
NIVEL2_PATH = f'/content/test_images/{[n for n in uploaded if "pag2-2" in n][0]}'
print(f'Nivel 1: {NIVEL1_PATH}')
print(f'Nivel 2: {NIVEL2_PATH}')


## Celda 3 — Ground truth propio (v13), mismo que en el notebook de MitUNet/Raster2Seq

Solo se necesitan los puntos de `muro` para este modelo (MLSTRUCT-FP/U-Net solo segmenta muros, no puertas/ventanas -- confirmado en el runbook).


In [ ]:
GT_NIVEL1_MURO = [
    (0.6745,0.0690),(0.8025,0.0695),(0.2194,0.0714),(0.9430,0.2004),
    (0.3298,0.3320),(0.6156,0.3980),(0.8771,0.4135),(0.2410,0.3166),
    (0.5757,0.4779),(0.7996,0.5415),(0.2433,0.7122),(0.6906,0.5737),
    (0.3682,0.5746),(0.2377,0.5927),(0.5864,0.6247),(0.3304,0.4661),
    (0.2451,0.4666),(0.5792,0.7129),(0.6978,0.4195),(0.8001,0.4407),
    (0.6208,0.4540),
]

GT_NIVEL2_MURO = [
    (0.6211,0.3170),(0.7080,0.3170),(0.3155,0.3123),(0.6021,0.3611),
    (0.3036,0.4037),(0.8541,0.4164),(0.6863,0.4735),(0.2141,0.4743),
    (0.4016,0.4809),(0.6955,0.5734),(0.5331,0.5754),(0.7816,0.5899),
    (0.2150,0.6300),(0.5023,0.6406),(0.5048,0.6911),(0.4461,0.7307),
    (0.2384,0.3123),(0.2398,0.5098),(0.3078,0.3421),(0.5023,0.5966),
    (0.3618,0.5098),(0.4179,0.5261),(0.3450,0.5696),(0.4105,0.4296),
    (0.5698,0.4300),(0.2607,0.5696),(0.4185,0.5726),
]

print(f'GT Nivel 1: {len(GT_NIVEL1_MURO)} puntos de muro')
print(f'GT Nivel 2: {len(GT_NIVEL2_MURO)} puntos de muro')


## Celda 4 — Descargar el checkpoint pre-entrenado `no_rot_256_50` (Google Drive)

Link confirmado leyendo el README real del repo (2026-08-06): `15ufkjoWOFyT0Cm-MEc9zQJCDJIooOgh7`. Formato exacto del contenido (zip de carpeta `.session/`, o pesos sueltos) sin confirmar hasta bajarlo -- si `gdown` trae un `.zip`, descomprimir y revisar la estructura antes de la Celda 6.


In [ ]:
%cd /content
!gdown --id 15ufkjoWOFyT0Cm-MEc9zQJCDJIooOgh7 -O checkpoint_no_rot_256_50.zip
!unzip -o checkpoint_no_rot_256_50.zip -d /content/checkpoint_no_rot_256_50
!find /content/checkpoint_no_rot_256_50 -maxdepth 3


## Celda 5 — Cargar el modelo y el checkpoint

Basado en la lectura directa de `_model.py`/`_fp_unet.py` (2026-08-06): `UNETFloorPhotoModel(data=None, name=..., image_shape=(256,256,1))` + `load_session(ruta)`. **La ruta exacta a pasarle a `load_session` depende de la estructura real que muestre la Celda 4** -- ajustar `SESSION_PATH` abajo con lo que aparezca ahí (el notebook oficial usa el patrón `.session/model_{nombre}`, sin confirmar si aplica igual al checkpoint descargado suelto).


In [ ]:
import sys
import types

# Fix real 2026-08-06 (tercera vuelta): ImportError: cannot import name
# 'distributed_file_utils' from 'tensorflow.python.distribute' -- un callback
# de logging para ENTRENAMIENTO (_tensorboardv2.py, que no usamos para pura
# inferencia) importa una API interna de TF que existia en 2023 y ya no esta
# en la version de TF que trae Colab. En vez de perseguir la ruta interna
# correcta de la version actual de TF (cambia constantemente, no vale la
# pena), se inyecta un modulo falso vacio para esa importacion puntual --
# nunca se llama nada de TensorBoard durante la inferencia.
sys.modules['tensorflow.python.distribute.distributed_file_utils'] = types.ModuleType('distributed_file_utils')
sys.path.insert(0, '/content/MLStructFP_benchmarks')
sys.path.insert(0, '/content/MLSTRUCT-FP')  # libreria base, por si los imports internos la necesitan
from MLStructFP_benchmarks.ml.model.architectures import UNETFloorPhotoModel

model = UNETFloorPhotoModel(data=None, name='no_rot_256_50', image_shape=(256, 256, 1))

# AJUSTAR esta ruta segun lo que haya mostrado el 'find' de la Celda 4:
SESSION_PATH = '/content/checkpoint_no_rot_256_50'
model.load_session(SESSION_PATH)
print('Sesion cargada OK')


## Celda 6 — Inferencia sobre Nivel 1 y Nivel 2, comparar contra el ground truth de muros

Mismo patron de verificacion visual + muestreo de puntos GT ya usado con MitUNet. Imagen en escala de grises, 256x256, `predict()` maneja la normalizacion 0-1 internamente segun el codigo fuente real (recibe uint8).


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def correr_mlstructfp(img_path, gt_muro_rel, nivel_nombre):
    img_bgr = cv2.imread(img_path)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_256 = cv2.resize(img_gray, (256, 256))
    x = img_256.reshape(1, 256, 256, 1).astype('uint8')

    pred = model.predict(x)
    pred = np.array(pred).reshape(256, 256)
    mask = (pred > 0.5).astype(np.uint8)

    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    axes[0].imshow(img_256, cmap='gray'); axes[0].set_title('Original (256x256)'); axes[0].axis('off')
    axes[1].imshow(mask, cmap='gray'); axes[1].set_title('Muros predichos (MLSTRUCT-FP)'); axes[1].axis('off')
    plt.tight_layout(); plt.show()

    aciertos = 0
    for (cx_rel, cy_rel) in gt_muro_rel:
        mx = int(cx_rel * 256)
        my = int(cy_rel * 256)
        ventana = mask[max(0,my-2):my+3, max(0,mx-2):mx+3]
        if ventana.size and ventana.max() == 1:
            aciertos += 1
    recall = aciertos / len(gt_muro_rel) if gt_muro_rel else float('nan')
    print(f'  [{nivel_nombre}] Puntos de muro GT dentro de la mascara predicha: {aciertos}/{len(gt_muro_rel)} (recall aprox: {recall:.1%})')
    return mask, recall

print('=== Nivel 1 ===')
mask_n1, recall_n1 = correr_mlstructfp(NIVEL1_PATH, GT_NIVEL1_MURO, 'Nivel 1')
print('\n=== Nivel 2 ===')
mask_n2, recall_n2 = correr_mlstructfp(NIVEL2_PATH, GT_NIVEL2_MURO, 'Nivel 2')
